# Getting RAM down
# DO NOT RUN!!!!! 

### unless you're curious. Takes up about 500-700mb of RAM  

In [ ]:
import pandas as pd

#Initial load letting pandas guess
df = pd.read_csv('logbook_assignment1.csv')

print(df.shape)
print(df.dtypes)
print(df.head())

# Memory usage BEFORE
mem_before = df.memory_usage(deep=True)
print(mem_before)
print(f"Total memory (before): {mem_before.sum() / 1024**2:.2f} MB")

In [ ]:
#Inspect columns
for col in df.columns:
    print(col, '->', df[col].dtype)
    print(df[col].head(3), '\n')

In [ ]:
#Re-read with explicit dtypes

#Columns that should be parsed as dates
date_cols = ['date_fueled', 'date_captured']

#Columns that are numeric
numeric_cols = ['odometer', 'gallons', 'cost_per_gallon', 'total_spent', 'mpg', 'miles']

#Columns that make sense as categoricals
category_cols = ['user_url']

df = pd.read_csv(
    'logbook_assignment1.csv',
    parse_dates=date_cols,
    dayfirst=True,          
)

#Clean and convert numeric columns (stripping symbols/commas if present, then cast)
for col in numeric_cols:
    df[col] = (
        df[col]
        .astype(str)
        .str.replace(r'[^\d.\-]', '', regex=True)
        .replace('', pd.NA)
    )
    df[col] = pd.to_numeric(df[col], errors='coerce')

#Downcast numeric types to smaller footprints
for col in numeric_cols:
    if pd.api.types.is_float_dtype(df[col]):
        df[col] = pd.to_numeric(df[col], downcast='float')
    elif pd.api.types.is_integer_dtype(df[col]):
        df[col] = pd.to_numeric(df[col], downcast='integer')

#Convert repeated string columns to categorical
for col in category_cols:
    df[col] = df[col].astype('category')

### Real work starts here

In [ ]:
import pandas as pd

df = pd.read_csv('logbook_assignment1.csv', low_memory=False)

# ---- Dates ----
date_cols = ['date_fueled', 'date_captured']
for col in date_cols:
    df[col] = pd.to_datetime(df[col], format='%b %d %Y', errors='coerce')

# ---- odometer / miles ----
for col in ['odometer', 'miles']:
    df[col] = (
        df[col].astype(str)
        .str.strip()
        .str.replace(',', '', regex=False)
    )
    df[col] = pd.to_numeric(df[col], errors='coerce')
    df[col] = pd.to_numeric(df[col], downcast='float')

# ---- cost_per_gallon / total_spent: vectorized currency split ----
for col in ['cost_per_gallon', 'total_spent']:
    extracted = df[col].astype(str).str.extract(r'^([^\d]*)([\d,]+\.?\d*)')
    currency = extracted[0].str.strip().replace('', None)
    amount = extracted[1].str.replace(',', '', regex=False)

    df[f'{col}_currency'] = currency.astype('category')
    df[col] = pd.to_numeric(amount, errors='coerce')
    df[col] = pd.to_numeric(df[col], downcast='float')

# ---- gallons / mpg ----
for col in ['gallons', 'mpg']:
    df[col] = pd.to_numeric(df[col], errors='coerce')
    df[col] = pd.to_numeric(df[col], downcast='float')

# ---- user_url ----
df['user_url'] = df['user_url'].astype('category')

# ---- Sanity checks ----
print(df.dtypes)
print(f"\nTotal memory: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
print("\nMissing values after cleaning:")
print(df.isna().sum())
print("\nCurrencies found:")
print(df['cost_per_gallon_currency'].value_counts())
print(df['total_spent_currency'].value_counts())